#DAY 5 Databricks Challenge

###PRACTISE TASK 1. Incremental MERGE (MOST IMPORTANT)

####Ensuring the data present in events as delta format exists

In [0]:
spark.read.format("delta") \
    .load("/Volumes/workspace/default/delta/events") \
    .show(5)

####Creating new incremental data to check the MERGE

In [0]:
from datetime import datetime
from pyspark.sql.types import *

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

updates_data = [
    (datetime(2025, 1, 12, 10, 0), "purchase", 10101, 2053013555631882655,
     "electronics.smartphone", "apple", 999.0, 70001, "session_new_01"),

    # duplicate record (should update, not insert)
    (datetime(2025, 1, 10, 10, 0), "view", 10101, 2053013555631882655,
     "electronics.smartphone", "samsung", 799.0, 50001, "session_001")
]

updates = spark.createDataFrame(updates_data, schema)
updates.show(truncate=False)


####Merging the data into events

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/delta/events"
)

deltaTable.alias("t").merge(
    updates.alias("s"),
    """
    t.user_session = s.user_session
    AND t.event_time = s.event_time
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


###Ensuring the newly added records exist in events using filtering condition

In [0]:
spark.read.format("delta") \
    .load("/Volumes/workspace/default/delta/events") \
    .filter("user_session IN ('session_new_01','session_001')") \
    .show(truncate=False)


#**************************************************

###PRACTISE TASK 2. Time Travel (Version History)

####Checking Table History

In [0]:
%sql 
DESCRIBE HISTORY events_table

####Querying an older version

In [0]:
old_version = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/delta/events")

old_version.show(5)


####Comparing it with new version using count of rows

In [0]:
latest = spark.read.format("delta") \
    .load("/Volumes/workspace/default/delta/events")

latest.count(), old_version.count()

#Look carefully at the result. You can find the new entries added in the latest one compared to original one.



####We can also query the data absed on the timestamp similar to version
####Easy when we want to know the updates that happened after a particular time
####Look at the count each timestamp returns

In [0]:
yesterday = spark.read.format("delta") \
    .option("timestampAsOf", "2026-01-12 15:13:44.0") \
    .load("/Volumes/workspace/default/delta/events")

yesterday.count()

In [0]:


today = spark.read.format("delta") \
    .option("timestampAsOf", "2026-01-11 14:58:49.0") \
    .load("/Volumes/workspace/default/delta/events")

today.count()


#**************************************************

###PRACTISE TASK 3. OPTIMIZE & ZORDER

####Run Optimize

In [0]:
%sql
OPTIMIZE events_table;


####OPTIMIZE with ZORDER

In [0]:
%sql
OPTIMIZE events_table
ZORDER BY (event_type, user_id);



#### Testing Performance

In [0]:
%sql
SELECT *
FROM events_table
WHERE event_type = 'view'
AND user_id = 525096345

-- Just to show the fast the query runs

#**************************************************


###PRACTISE TASK 4. VACUUM (Cleanup Old Files)

####VACUUM is generally used with 7 days timegap which means the data before a week is kept alive

In [0]:
%sql
VACUUM events_table RETAIN 168 HOURS;
--WE SHOULD NOT KEEP RETAIN 0 HOURS WHICH DELETES ALL HISTORY AND TIMETRAVEL IS NOT POSSIBLE

In [0]:
%sql
DESCRIBE HISTORY events_table;

--lookout for recent change and you can see Operation parameters milliseconds as 604800000 which is equivalent to 168hours.




## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 